# RGB Defect Detection

Object detection of visible solar-panel defects from RGB images using YOLO26m.

**Task:** RGB defect detection with bounding boxes  
**Classes:** bird-drop, clean, dusty, electrical-damage, physical-damage, snow-capped  
**Final artifact:** `best.pt` → ONNX

This notebook follows the same direct, Ultralytics-oriented structure used in the panel-segmentation notebook. Task-specific differences are limited to detection annotations, box metrics, and detection evaluation.

## 01. Project Definition

Input:
- RGB solar-panel images
- YOLO-compatible bounding-box annotations

Output:
- Defect class
- Bounding box
- Confidence score

Evaluation:
- Precision
- Recall
- mAP50
- mAP50-95
- Per-class performance
- Small-defect behavior
- Error analysis
- Unseen-image testing

## 02. Dataset Configuration

In [5]:
from pathlib import Path
import json
import yaml
import hashlib
import random
import shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter, defaultdict
from PIL import Image
from ultralytics import YOLO
from ultralytics.data.utils import check_det_dataset

In [6]:
DATASET_DIR = Path(r"D:\rgb_defect_pannel_db")

TRAIN_DIR = DATASET_DIR / "train"
VALID_DIR = DATASET_DIR / "valid"
TEST_DIR = DATASET_DIR / "test"

TRAIN_IMAGES = TRAIN_DIR / "images"
VALID_IMAGES = VALID_DIR / "images"
TEST_IMAGES = TEST_DIR / "images"

TRAIN_LABELS = TRAIN_DIR / "labels"
VALID_LABELS = VALID_DIR / "labels"
TEST_LABELS = TEST_DIR / "labels"

SOURCE_DATA_YAML = DATASET_DIR / "data.yaml"

# Where the processed/prepared dataset will be created
PREPARED_DATASET_DIR = Path(r"D:\rgb_defect_detection_dataset")

DATA_YAML = PREPARED_DATASET_DIR / "data.yaml"

CLASS_NAMES = [
    "bird-drop",
    "clean",
    "dusty",
    "electrical-damage",
    "physical-damage",
    "snow-capped"
]

print("Dataset:", DATASET_DIR)
print("Prepared dataset:", PREPARED_DATASET_DIR)
print("Classes:", CLASS_NAMES)

Dataset: D:\rgb_defect_pannel_db
Prepared dataset: D:\rgb_defect_detection_dataset
Classes: ['bird-drop', 'clean', 'dusty', 'electrical-damage', 'physical-damage', 'snow-capped']


## 03. Dataset Inspection

In [7]:
def count_files(directory, suffix=None):
    files = [p for p in directory.iterdir() if p.is_file()]
    if suffix:
        files = [p for p in files if p.suffix.lower() == suffix]
    return len(files)

for name, image_dir, label_dir in [
    ("train", TRAIN_IMAGES, TRAIN_LABELS),

    ("valid", VALID_IMAGES, VALID_LABELS),
    ("test", TEST_IMAGES, TEST_LABELS),
]:
    print(
        f"{name:>5} | images: {count_files(image_dir):4d} | "
        f"labels: {count_files(label_dir, '.txt'):4d}"
    )

print("Total images:", sum(count_files(d) for d in [TRAIN_IMAGES, VALID_IMAGES, TEST_IMAGES]))
print("Total labels:", sum(count_files(d, ".txt") for d in [TRAIN_LABELS, VALID_LABELS, TEST_LABELS]))

train | images: 5752 | labels:  678
valid | images:  678 | labels:  678
 test | images:  411 | labels:  411
Total images: 6841
Total labels: 1767


In [10]:
from PIL import Image
from ultralytics import YOLO

In [11]:
image_shapes = []

for image_path in TRAIN_IMAGES.iterdir():
    if image_path.is_file():
        try:
            with Image.open(image_path) as image:
                image_shapes.append(image.size)
        except Exception:
            pass

shape_counts = Counter(image_shapes)

print("Training image shapes:")
for shape, count in shape_counts.most_common():
    print(f"{shape}: {count}")

Training image shapes:
(640, 640): 1567


## 04. Dataset Analysis

In [13]:
print("YAML path:", SOURCE_DATA_YAML)
print("Exists:", SOURCE_DATA_YAML.exists())
print("File size:", SOURCE_DATA_YAML.stat().st_size, "bytes")

with open(SOURCE_DATA_YAML, "r", encoding="utf-8") as file:
    content = file.read()

print("\n===== data.yaml CONTENT =====")
print(repr(content))
print("===== END =====")

YAML path: D:\rgb_defect_pannel_db\data.yaml
Exists: True
File size: 0 bytes

===== data.yaml CONTENT =====
''
===== END =====


In [12]:
with open(SOURCE_DATA_YAML, "r") as file:
    source_data = yaml.safe_load(file)

print("Number of classes:", source_data["nc"])
print("Class names:")
for index, name in enumerate(source_data["names"]):
    print(f"{index}: {name}")

TypeError: 'NoneType' object is not subscriptable

In [ ]:
def annotation_statistics(label_dir):
    class_counter = Counter()
    bbox_widths = []
    bbox_heights = []
    bbox_areas = []
    format_counts = Counter()
    empty_files = []

    for label_path in label_dir.glob("*.txt"):
        lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]

        if not lines:
            empty_files.append(label_path.name)
            continue

        for line in lines:
            values = line.split()
            format_counts[len(values)] += 1

            if len(values) == 5:
                class_id = int(values[0])
                width = float(values[3])
                height = float(values[4])

                class_counter[class_id] += 1
                bbox_widths.append(width)
                bbox_heights.append(height)
                bbox_areas.append(width * height)

    return class_counter, bbox_widths, bbox_heights, bbox_areas, format_counts, empty_files

class_counts, bbox_widths, bbox_heights, bbox_areas, format_counts, empty_labels = (
    annotation_statistics(TRAIN_LABELS)
)

print("Training annotation class distribution:")
for class_id, count in sorted(class_counts.items()):
    print(f"{class_id}: {CLASS_NAMES[class_id]:<20} {count}")

print("\nAnnotation line formats:")
print(dict(sorted(format_counts.items())))

print("\nEmpty training label files:", len(empty_labels))

Training annotation class distribution:
0: bird-drop            4193
1: clean                1324
2: dusty                1090
3: electrical-damage    635
4: physical-damage      4003
5: snow-capped          2309

Annotation line formats:
{5: 13554, 9: 53, 11: 269, 13: 161, 15: 122, 17: 62, 19: 40, 21: 28, 23: 20, 25: 9, 27: 12, 29: 12, 31: 6, 33: 8, 35: 5, 37: 2, 39: 2}

Empty training label files: 158


## 05. Annotation Verification

In [ ]:
def validate_annotations():
    result = {
        "missing_labels": [],
        "missing_images": [],
        "empty_labels": [],
        "invalid_class_ids": [],
        "invalid_bbox_values": [],
        "non_detection_formats": []
    }

    image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    for split_name, image_dir, label_dir in [
        ("train", TRAIN_IMAGES, TRAIN_LABELS),
        ("valid", VALID_IMAGES, VALID_LABELS),
        ("test", TEST_IMAGES, TEST_LABELS),
    ]:
        image_files = {
            p.stem for p in image_dir.iterdir()
            if p.is_file() and p.suffix.lower() in image_extensions
        }
        label_files = {
            p.stem: p for p in label_dir.glob("*.txt")
        }

        for stem in image_files - set(label_files):
            result["missing_labels"].append(f"{split_name}/{stem}")

        for stem in set(label_files) - image_files:
            result["missing_images"].append(f"{split_name}/{stem}")

        for stem, label_path in label_files.items():
            lines = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]

            if not lines:
                result["empty_labels"].append(f"{split_name}/{label_path.name}")
                continue

            for line_number, line in enumerate(lines, 1):
                values = line.split()

                if len(values) != 5:
                    result["non_detection_formats"].append(
                        f"{split_name}/{label_path.name}: line {line_number} ({len(values)} values)"
                    )
                    continue

                try:
                    class_id = int(values[0])
                    box = [float(v) for v in values[1:]]
                except ValueError:
                    result["invalid_bbox_values"].append(
                        f"{split_name}/{label_path.name}: line {line_number}"
                    )
                    continue

                if not 0 <= class_id < len(CLASS_NAMES):
                    result["invalid_class_ids"].append(
                        f"{split_name}/{label_path.name}: class_id={class_id}"
                    )

                if any(v < 0 or v > 1 for v in box):
                    result["invalid_bbox_values"].append(
                        f"{split_name}/{label_path.name}: line {line_number}"
                    )

    return result

validation = validate_annotations()

print("Missing labels:", len(validation["missing_labels"]))
print("Missing images:", len(validation["missing_images"]))
print("Empty labels:", len(validation["empty_labels"]))
print("Invalid class IDs:", len(validation["invalid_class_ids"]))
print("Invalid bbox values:", len(validation["invalid_bbox_values"]))
print("Non-detection annotation lines:", len(validation["non_detection_formats"]))

Missing labels: 0
Missing images: 0
Empty labels: 190
Invalid class IDs: 0
Invalid bbox values: 0
Non-detection annotation lines: 968


In [ ]:
def show_annotation_formats(label_dir, limit=5):
    shown = 0
    for label_path in label_dir.glob("*.txt"):
        for line in label_path.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            values = line.split()
            if len(values) != 5:
                print(label_path.name, "->", len(values), "values")
                print(line)
                shown += 1
                if shown >= limit:
                    return

show_annotation_formats(TRAIN_LABELS)

Clean-123-_jpg.rf.20786b707b3c6d902dd0626f62c969cd.txt -> 13 values
1 1 0.28003961093749996 0.17830427499999998 0.80828480625 0.28415084375 0.9958271265624999 0.7475441703125 0.9903507578124999 1 0.6559960734375 1 0.28003961093749996
Clean-123-_jpg.rf.20786b707b3c6d902dd0626f62c969cd.txt -> 11 values
1 0.9152555484375 0.253952803125 0.805370871875 0.15016420625 0.0032642671875000003 0.3705397078125 0.12017962812500001 0.6791711796875 0.9152555484375 0.253952803125
Clean-123-_jpg.rf.20786b707b3c6d902dd0626f62c969cd.txt -> 11 values
1 0 0.340169621875 0.7310494265624999 0.1621157234375 0.6472127546875 0.0728602390625 0 0.14301191875000002 0 0.340169621875
Dust-170-_jpg.rf.a39443fa133f77a71245be348433fd0b.txt -> 15 values
2 1 0.2325228484375 0.3387391421875 0.221795809375 0 0.43795900000000004 0 1 0.214329178125 1 1 0.9393660593750001 1 0.2325228484375
Dust-123-_jpg.rf.71b0c71f3ba16fccec581d798dc14d17.txt -> 15 values
2 0.5641095203125001 0 0.1808042171875 0.4475468125 0.125 0.81279723281

## 06. Data Leakage / Duplicate Check

In [ ]:
def image_hash(image_path):
    h = hashlib.md5()
    with open(image_path, "rb") as file:
        for chunk in iter(lambda: file.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

hash_map = defaultdict(list)

for split_name, image_dir in [
    ("train", TRAIN_IMAGES),
    ("valid", VALID_IMAGES),
    ("test", TEST_IMAGES),
]:
    for image_path in image_dir.iterdir():
        if image_path.is_file():
            hash_map[image_hash(image_path)].append((split_name, image_path))

duplicate_groups = [
    files for files in hash_map.values()
    if len(files) > 1
]

cross_split_duplicates = [
    files for files in duplicate_groups
    if len({split for split, _ in files}) > 1
]

print("Duplicate groups:", len(duplicate_groups))
print("Cross-split duplicate groups:", len(cross_split_duplicates))

for group in cross_split_duplicates:
    print("\nDuplicate group:")
    for split, path in group:
        print(split, path.name)

Duplicate groups: 43
Cross-split duplicate groups: 1

Duplicate group:
train Bird-87-_jpg.rf.0c7a3c46376b9c587dacb55896515c9d.jpg
test Bird-81-_jpg.rf.e7f2d1d6e9197adbec1f410195f824f6.jpg


## 07. Data Preparation

The source dataset is not regenerated unnecessarily. The required preparation is only:

1. Preserve existing 5-value YOLO detection annotations.
2. Convert polygon-style annotation lines to bounding boxes because this model is detection, not segmentation.
3. Preserve empty label files.
4. Remove only confirmed cross-split duplicate image copies from the training split.
5. Verify the resulting image/label pairing and annotation format.

In [ ]:
def polygon_to_bbox(values):
    class_id = int(values[0])
    coordinates = [float(v) for v in values[1:]]

    x_values = coordinates[0::2]
    y_values = coordinates[1::2]

    x_min, x_max = min(x_values), max(x_values)
    y_min, y_max = min(y_values), max(y_values)

    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min

    return f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}"

def prepare_label_file(source_label, destination_label):
    lines = [
        line.strip()
        for line in source_label.read_text().splitlines()
        if line.strip()
    ]

    prepared = []

    for line in lines:
        values = line.split()

        if len(values) == 5:
            prepared.append(line)
        elif len(values) > 5 and len(values[1:]) % 2 == 0:
            prepared.append(polygon_to_bbox(values))

    destination_label.parent.mkdir(parents=True, exist_ok=True)
    destination_label.write_text("\n".join(prepared))

def prepare_split(source_image_dir, source_label_dir, destination_split, excluded_files):
    destination_images = destination_split / "images"
    destination_labels = destination_split / "labels"

    destination_images.mkdir(parents=True, exist_ok=True)
    destination_labels.mkdir(parents=True, exist_ok=True)

    for image_path in source_image_dir.iterdir():
        if not image_path.is_file():
            continue
        if image_path.name in excluded_files:
            continue

        shutil.copy2(image_path, destination_images / image_path.name)

    for label_path in source_label_dir.glob("*.txt"):
        if label_path.stem in {Path(name).stem for name in excluded_files}:
            continue

        prepare_label_file(
            label_path,
            destination_labels / label_path.name
        )

In [ ]:
excluded_train_files = set()

for group in cross_split_duplicates:
    train_files = [path for split, path in group if split == "train"]
    other_files = [path for split, path in group if split != "train"]

    if train_files and other_files:
        excluded_train_files.add(train_files[0].name)

print("Training files excluded because of cross-split duplication:")
for name in sorted(excluded_train_files):
    print(name)

Training files excluded because of cross-split duplication:
Bird-87-_jpg.rf.0c7a3c46376b9c587dacb55896515c9d.jpg


In [ ]:
# Build the prepared detection dataset.

if PREPARED_DATASET_DIR.exists():
    shutil.rmtree(PREPARED_DATASET_DIR)

prepare_split(
    TRAIN_IMAGES,
    TRAIN_LABELS,
    PREPARED_DATASET_DIR / "train",
    excluded_train_files
)

prepare_split(
    VALID_IMAGES,
    VALID_LABELS,
    PREPARED_DATASET_DIR / "valid",
    set()
)

prepare_split(
    TEST_IMAGES,
    TEST_LABELS,
    PREPARED_DATASET_DIR / "test",
    set()
)

print("Prepared dataset created:", PREPARED_DATASET_DIR)

Prepared dataset created: /home/hraj/Workspace/Solor_defect_DB/Database/rgb_defect_detection_dataset


In [ ]:
def validate_prepared_dataset(dataset_dir):
    for split in ["train", "valid", "test"]:
        image_dir = dataset_dir / split / "images"
        label_dir = dataset_dir / split / "labels"

        images = {p.stem for p in image_dir.iterdir() if p.is_file()}
        labels = {p.stem for p in label_dir.glob("*.txt")}

        invalid = []

        for label_path in label_dir.glob("*.txt"):
            for line_number, line in enumerate(label_path.read_text().splitlines(), 1):
                if not line.strip():
                    continue
                values = line.split()
                if len(values) != 5:
                    invalid.append(f"{label_path.name}: line {line_number}")

        print(
            f"{split:>5} | images={len(images):4d} | labels={len(labels):4d} | "
            f"missing_labels={len(images-labels):2d} | "
            f"missing_images={len(labels-images):2d} | "
            f"invalid_annotations={len(invalid):2d}"
        )

validate_prepared_dataset(PREPARED_DATASET_DIR)

train | images=5751 | labels=5751 | missing_labels= 0 | missing_images= 0 | invalid_annotations= 0
valid | images= 821 | labels= 821 | missing_labels= 0 | missing_images= 0 | invalid_annotations= 0
 test | images= 411 | labels= 411 | missing_labels= 0 | missing_images= 0 | invalid_annotations= 0


## 08. Training Dataset Configuration

In [ ]:
data = {
    "path": str(PREPARED_DATASET_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 6,
    "names": CLASS_NAMES
}

with open(DATA_YAML, "w") as file:
    yaml.safe_dump(data, file, sort_keys=False)

print(DATA_YAML)
print(check_det_dataset(str(DATA_YAML)))

/home/hraj/Workspace/Solor_defect_DB/Database/rgb_defect_detection_dataset/data.yaml
{'path': PosixPath('/home/hraj/Workspace/Solor_defect_DB/Database/rgb_defect_detection_dataset'), 'train': '/home/hraj/Workspace/Solor_defect_DB/Database/rgb_defect_detection_dataset/train/images', 'val': '/home/hraj/Workspace/Solor_defect_DB/Database/rgb_defect_detection_dataset/valid/images', 'test': '/home/hraj/Workspace/Solor_defect_DB/Database/rgb_defect_detection_dataset/test/images', 'nc': 6, 'names': {0: 'bird-drop', 1: 'clean', 2: 'dusty', 3: 'electrical-damage', 4: 'physical-damage', 5: 'snow-capped'}, 'yaml_file': '/home/hraj/Workspace/Solor_defect_DB/Database/rgb_defect_detection_dataset/data.yaml', 'channels': 3}


## 09. Model Definition

In [ ]:
MODEL_NAME = "yolo26m.pt"

model = YOLO(MODEL_NAME)

print("Model loaded:", MODEL_NAME)

Model loaded: yolo26m.pt


## 10. Baseline Training

In [ ]:
IMG_SIZE = 640
BATCH_SIZE = 4
EPOCHS = 50
WORKERS = 2
DEVICE = "cpu"

AUGMENTATION = {
    "hsv_h": 0.015,
    "hsv_s": 0.5,
    "hsv_v": 0.3,
    "degrees": 5.0,
    "translate": 0.05,
    "scale": 0.2,
    "fliplr": 0.5,
    "flipud": 0.0,
    "mosaic": 1.0,
    "mixup": 0.0
}

PROJECT_DIR = Path.home() / "Workspace" / "Solor_defect_DB" / "Solar_Inspection" / "Experiment" / "runs" / "rgb_defect_detection"
RUN_NAME = "baseline"

print("Epochs:", EPOCHS)
print("Image size:", IMG_SIZE)
print("Batch:", BATCH_SIZE)
print("Device:", DEVICE)

Epochs: 50
Image size: 640
Batch: 4
Device: cpu


In [ ]:
result = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMG_SIZE,
    workers=WORKERS,
    device=DEVICE,

    hsv_h=AUGMENTATION["hsv_h"],
    hsv_s=AUGMENTATION["hsv_s"],
    hsv_v=AUGMENTATION["hsv_v"],
    degrees=AUGMENTATION["degrees"],
    translate=AUGMENTATION["translate"],
    scale=AUGMENTATION["scale"],
    fliplr=AUGMENTATION["fliplr"],
    flipud=AUGMENTATION["flipud"],
    mosaic=AUGMENTATION["mosaic"],
    mixup=AUGMENTATION["mixup"],

    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=True
)

New https://pypi.org/project/ultralytics/8.4.157 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.152 🚀 Python-3.10.12 torch-2.14.0+cu130 CPU (13th Gen Intel Core i5-13420H)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/hraj/Workspace/Solor_defect_DB/Database/rgb_defect_detection_dataset/data.yaml, degrees=5.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.5, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01,

## 11. Training Analysis

In [ ]:
RUN_DIR = PROJECT_DIR / "baseline-4"

RESULTS_CSV = RUN_DIR / "results.csv"
RESULTS_PLOT = RUN_DIR / "results.png"
BEST_MODEL = RUN_DIR / "weights" / "best.pt"

print("Run directory:", RUN_DIR)
print("Best model:", BEST_MODEL)
print("Exists:", BEST_MODEL.exists())

Run directory: /home/hraj/Workspace/Solor_defect_DB/Solar_Inspection/Experiment/runs/rgb_defect_detection/baseline-4
Best model: /home/hraj/Workspace/Solor_defect_DB/Solar_Inspection/Experiment/runs/rgb_defect_detection/baseline-4/weights/best.pt
Exists: False


In [ ]:
if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)

    print("Epochs recorded:", len(results_df))
    display_columns = [
        column for column in [
            "epoch",
            "train/box_loss",
            "train/cls_loss",
            "metrics/precision(B)",
            "metrics/recall(B)",
            "metrics/mAP50(B)",
            "metrics/mAP50-95(B)"
        ]
        if column in results_df.columns
    ]

    display(results_df[display_columns].tail())

    if "metrics/mAP50(B)" in results_df.columns:
        print("Best mAP50:", results_df["metrics/mAP50(B)"].max())

    if "metrics/mAP50-95(B)" in results_df.columns:
        print("Best mAP50-95:", results_df["metrics/mAP50-95(B)"].max())
else:
    print("results.csv not found:", RESULTS_CSV)

results.csv not found: /home/hraj/Workspace/Solor_defect_DB/Solar_Inspection/Experiment/runs/rgb_defect_detection/baseline-4/results.csv


## 12. Validation / Metrics

In [ ]:
best_model = YOLO(str(BEST_MODEL))

validation = best_model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True
)

print("Precision :", validation.box.p)
print("Recall    :", validation.box.r)
print("mAP50     :", validation.box.map50)
print("mAP50-95  :", validation.box.map)

FileNotFoundError: [Errno 2] No such file or directory: '/home/hraj/Workspace/Solor_defect_DB/Solar_Inspection/Experiment/runs/rgb_defect_detection/baseline-4/weights/best.pt'

## 13. Per-Class Evaluation

In [ ]:
print("Per-class detection performance")

for index, class_name in enumerate(CLASS_NAMES):
    print(
        f"{index}: {class_name:<20} "
        f"Precision={validation.box.p[index]:.4f}  "
        f"Recall={validation.box.r[index]:.4f}  "
        f"mAP50={validation.box.ap50[index]:.4f}  "
        f"mAP50-95={validation.box.ap[index]:.4f}"
    )

## 14. Prediction Visualization

In [ ]:
VAL_IMAGES = PREPARED_DATASET_DIR / "valid" / "images"

val_predictions = best_model.predict(
    source=str(VAL_IMAGES),
    imgsz=IMG_SIZE,
    device=DEVICE,
    conf=0.25,
    verbose=False
)

print("Validation images predicted:", len(val_predictions))

In [ ]:
num_images = min(6, len(val_predictions))

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, ax in enumerate(axes):
    ax.axis("off")

    if i < num_images:
        plotted = val_predictions[i].plot()
        ax.imshow(cv2.cvtColor(plotted, cv2.COLOR_BGR2RGB))
        ax.set_title(Path(val_predictions[i].path).name)

plt.tight_layout()
plt.show()

## 15. Small-Defect Evaluation

In [ ]:
SMALL_AREA_THRESHOLD = 0.01

def get_small_boxes(label_dir):
    small_boxes = []
    total_boxes = 0

    for label_path in label_dir.glob("*.txt"):
        for line in label_path.read_text().splitlines():
            values = line.split()

            if len(values) != 5:
                continue

            width = float(values[3])
            height = float(values[4])
            area = width * height

            total_boxes += 1

            if area < SMALL_AREA_THRESHOLD:
                small_boxes.append({
                    "file": label_path.name,
                    "class_id": int(values[0]),
                    "area": area
                })

    return small_boxes, total_boxes

small_boxes, total_boxes = get_small_boxes(
    PREPARED_DATASET_DIR / "valid" / "labels"
)

print("Validation ground-truth boxes:", total_boxes)
print("Small boxes:", len(small_boxes))
print("Small-box ratio:", round(len(small_boxes) / total_boxes, 4) if total_boxes else 0)

Small-defect analysis here identifies the small-object population in the ground truth. The standard Ultralytics validation metrics remain the primary quantitative detection metrics; small-object behavior is additionally inspected through the prediction visualizations and error analysis.

## 16. Error Analysis

In [ ]:
def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - inter_area

    return inter_area / union if union > 0 else 0.0

def normalized_to_xyxy(values, width, height):
    x_center, y_center, box_w, box_h = map(float, values[1:5])

    x1 = (x_center - box_w / 2) * width
    y1 = (y_center - box_h / 2) * height
    x2 = (x_center + box_w / 2) * width
    y2 = (y_center + box_h / 2) * height

    return [x1, y1, x2, y2]

def analyze_image_errors(result, label_path, iou_threshold=0.5):
    image = cv2.imread(str(result.path))
    height, width = image.shape[:2]

    gt = []
    for line in label_path.read_text().splitlines():
        values = line.split()
        if len(values) == 5:
            gt.append({
                "class_id": int(values[0]),
                "box": normalized_to_xyxy(values, width, height)
            })

    predictions = []
    if result.boxes is not None:
        for box, cls, conf in zip(
            result.boxes.xyxy.cpu().numpy(),
            result.boxes.cls.cpu().numpy(),
            result.boxes.conf.cpu().numpy()
        ):
            predictions.append({
                "class_id": int(cls),
                "box": box.tolist(),
                "confidence": float(conf)
            })

    matched_gt = set()
    matched_pred = set()
    class_confusion = 0
    localization_errors = 0

    for pred_index, pred in enumerate(predictions):
        best_iou = 0
        best_gt_index = None

        for gt_index, target in enumerate(gt):
            if gt_index in matched_gt:
                continue

            iou = box_iou(pred["box"], target["box"])

            if iou > best_iou:
                best_iou = iou
                best_gt_index = gt_index

        if best_gt_index is not None and best_iou >= iou_threshold:
            matched_gt.add(best_gt_index)
            matched_pred.add(pred_index)

            if predictions[pred_index]["class_id"] != gt[best_gt_index]["class_id"]:
                class_confusion += 1
            elif best_iou < 0.75:
                localization_errors += 1

    false_positives = len(predictions) - len(matched_pred)
    false_negatives = len(gt) - len(matched_gt)

    return {
        "false_positives": false_positives,
        "false_negatives": false_negatives,
        "class_confusion": class_confusion,
        "localization_errors": localization_errors
    }

In [ ]:
error_summary = Counter()

for result in val_predictions:
    label_path = (
        PREPARED_DATASET_DIR
        / "valid"
        / "labels"
        / f"{Path(result.path).stem}.txt"
    )

    if label_path.exists():
        errors = analyze_image_errors(result, label_path)

        for key, value in errors.items():
            error_summary[key] += value

print("Validation error summary:")
for key, value in error_summary.items():
    print(f"{key}: {value}")

## 17. Threshold Analysis

In [ ]:
THRESHOLDS = [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]

threshold_results = []

for threshold in THRESHOLDS:
    metrics = best_model.val(
        data=str(DATA_YAML),
        split="val",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        conf=threshold,
        verbose=False
    )

    threshold_results.append({
        "confidence": threshold,
        "precision": float(metrics.box.p),
        "recall": float(metrics.box.r),
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map)
    })

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)

## 18. Model Tuning

**Skipped for V1.**

The baseline YOLO26m model is retained as the final model. No second training run or hyperparameter tuning is performed.

## 19. Final Test Evaluation

In [ ]:
TEST_IMAGES = PREPARED_DATASET_DIR / "test" / "images"

final_test = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True
)

print("Final Test Metrics")
print("------------------")
print("Precision :", final_test.box.p)
print("Recall    :", final_test.box.r)
print("mAP50     :", final_test.box.map50)
print("mAP50-95  :", final_test.box.map)

## 20. Real-World / Unseen Data Testing

In [ ]:
UNSEEN_DIR = (
    Path.home()
    / "Workspace"
    / "Solor_defect_DB"
    / "Database"
    / "unseen_rgb_images"
)

REAL_WORLD_OUTPUT = (
    Path.home()
    / "Workspace"
    / "Solor_defect_DB"
    / "Solar_Inspection"
    / "Experiment"
    / "runs"
    / "detect"
    / "real_world_testing"
)

print("Unseen image directory:", UNSEEN_DIR)
print("Exists:", UNSEEN_DIR.exists())

if UNSEEN_DIR.exists():
    unseen_results = best_model.predict(
        source=str(UNSEEN_DIR),
        imgsz=IMG_SIZE,
        device=DEVICE,
        conf=0.25,
        save=True,
        project=str(REAL_WORLD_OUTPUT),
        name="predictions",
        verbose=False
    )

    print("Unseen images predicted:", len(unseen_results))
    print("Output:", REAL_WORLD_OUTPUT / "predictions")
else:
    print("Add genuinely unseen RGB images to this directory before running the test.")

## 21. Model Export

In [ ]:
FINAL_MODEL = BEST_MODEL

print("Final model:", FINAL_MODEL)
print("Exists:", FINAL_MODEL.exists())

In [ ]:
onnx_path = best_model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    device=DEVICE
)

print("ONNX export completed.")
print("ONNX model:", onnx_path)
print("Exists:", Path(onnx_path).exists())

## 22. Final Conclusion

In [ ]:
print("FINAL RGB DEFECT DETECTION MODEL")
print("================================")
print("Model       :", FINAL_MODEL)
print("ONNX        :", onnx_path)
print()
print("Final Test Performance")
print("----------------------")
print("Precision   :", round(float(final_test.box.p), 4))
print("Recall      :", round(float(final_test.box.r), 4))
print("mAP50       :", round(float(final_test.box.map50), 4))
print("mAP50-95    :", round(float(final_test.box.map), 4))
print()
print("Classes")
print("-------")
for index, name in enumerate(CLASS_NAMES):
    print(f"{index}: {name}")